In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import fetch_olivetti_faces
from sklearn.preprocessing import OneHotEncoder
import logging

# Configure logging to overwrite "_.log" every run (mode="w")
logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler("q3a.log", mode="w"),  # "w" overwrites, "a" appends
        logging.StreamHandler()
    ],
    force=True  # Resets any existing logging handlers
)
logger = logging.getLogger(__name__)

faces = fetch_olivetti_faces(shuffle=False)
X = faces.images
y = faces.target

height = X.shape[1]
width = X.shape[2]

split_data = np.load("olivetti_split_indices.npz")

train_idx = split_data["train"]
val_idx = split_data["val"]
test_idx = split_data["test"]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

X_test_corrupted = X_test.copy()
X_test_corrupted[:, 24:40, 24:40] = 0.0

logger.info(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
logger.info(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
logger.info(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
logger.info(f"X_test_corrupted:  {X_test_corrupted.shape}, y_test:  {y_test.shape}")




X_train = X_train.reshape(-1, height * width)
X_val = X_val.reshape(-1, height * width)
X_test = X_test.reshape(-1, height * width)
X_test_corrupted = X_test_corrupted.reshape(-1, height * width)

N = len(y_test)
R = 30
classes = np.unique(y_train)
K = len(classes)
seed = 260007

encoder = OneHotEncoder(sparse_output=False, categories=[np.arange(K)])
e_targets = encoder.fit_transform(y_test.reshape(-1, 1))

def fit_samme_adaboost(X_flat, y, n_estimators, max_depth, seed):

    N = len(y)
    classes = np.unique(y)
    K = len(classes)
    top_k = 10

    w = np.ones(N, dtype=np.float64) / N

    estimators = []
    estimator_weights = []

    history = {
        "eps_ts": [],
        "alphas": [],
        "max_sample_weights": [],
        "top10_weights": [],
        "sample_weights_over_time": []
    }
    
    for t in range(n_estimators):
        clf = DecisionTreeClassifier(max_depth=max_depth, random_state=seed + t)
        clf.fit(X_flat, y, sample_weight=w)

        preds = clf.predict(X_flat)
        incorrect = (preds != y)

        eps_t = np.sum(w * incorrect) / np.sum(w)

        random_guess_error = 1.0 - (1.0 / K)
        if eps_t >= random_guess_error:
            if t == 0:
                alpha_t = 1e-10
                estimators.append(clf)
                estimator_weights.append(alpha_t)
            break

        if eps_t == 0:
            alpha_t = 100.0
        else:
            alpha_t = np.log((1.0 - eps_t) / eps_t) + np.log(K - 1)

        w_updated = w * np.exp(alpha_t * incorrect)

        w_next = w_updated / np.sum(w_updated)

        estimators.append(clf)
        estimator_weights.append(alpha_t)

        history["eps_ts"].append(eps_t)
        history["alphas"].append(alpha_t)
        history["max_sample_weights"].append(np.max(w_next))
        
        history["top10_weights"].append(np.sum(np.sort(w_next)[-top_k:]))
        history["sample_weights_over_time"].append(w_next.copy())

        w = w_next

    return estimators, estimator_weights, classes, history

def generate_resampled_indices(y_train, random_state):
    rng = np.random.default_rng(random_state)
    resampled_indices = []
    for cls_l in np.unique(y_train):
        cls_idx = np.where(y_train == cls_l)[0]
        chosen = rng.choice(cls_idx, size=len(cls_idx), replace=True)
        resampled_indices.extend(chosen)
    return np.array(resampled_indices)

def get_tree_proba(tree, X_flat):
    probas = tree.predict_proba(X_flat)
    if probas.shape[1] < K:
        full_p = np.zeros((X_flat.shape[0], K))
        full_p[:, tree.classes_] = probas
        return full_p
    return probas

def get_boosting_proba(estimators, estimator_weights, X_flat):
    N_samples = X_flat.shape[0]
    scores = np.zeros((N_samples, K))

    for clf, alpha in zip(estimators, estimator_weights):
        preds = clf.predict(X_flat)
        for k_idx, c in enumerate(classes):
            scores[:, k_idx] += alpha * (preds == c)

    total_alpha = np.sum(estimator_weights)
    return scores / total_alpha if total_alpha > 0 else scores / K

def compute_bias_and_variance(probs_r, e):
    p_bar = np.mean(probs_r, axis=0)
    bias = np.mean(np.sum((p_bar - e) ** 2, axis=1))
    diff = probs_r - p_bar[np.newaxis, :, :]
    variance = np.mean(np.sum(diff ** 2, axis=2))
    return bias, variance

single_tree_probs = np.zeros((R, N, K))
bagging_probs = np.zeros((R, N, K))
boosting_probs = np.zeros((R, N, K))

single_tree_accs = []
bagging_accs = []
boosting_accs = []

for r in range(R):
    logger.info(f"--- Iteration {r + 1}/{R} ---")
    resample_idx = generate_resampled_indices(y_train, seed + r)
    X_tr_r, y_tr_r = X_train[resample_idx], y_train[resample_idx]
    
    #Single Tree (Depth=None, min_samples_leaf=2)
    st = DecisionTreeClassifier(max_depth=None, min_samples_leaf=2, random_state=seed + r)
    st.fit(X_tr_r, y_tr_r)
    p_st = get_tree_proba(st, X_test)
    single_tree_probs[r] = p_st
    single_tree_accs.append(np.mean(np.argmax(p_st, axis=1) == y_test))
    
    #Bagged Model (B=200, Depth=None, min_samples_leaf=2)
    bag_trees = []
    b_rng = np.random.default_rng(seed + r)
    for b in range(200):
        boot_idx = b_rng.choice(len(y_tr_r), size=len(y_tr_r), replace=True)
        clf = DecisionTreeClassifier(max_depth=None, min_samples_leaf=2, random_state=seed + b)
        clf.fit(X_tr_r[boot_idx], y_tr_r[boot_idx])
        bag_trees.append(clf)
    all_p = np.array([get_tree_proba(clf, X_test) for clf in bag_trees])
    p_bag = np.mean(all_p, axis=0)
    bagging_probs[r] = p_bag
    bagging_accs.append(np.mean(np.argmax(p_bag, axis=1) == y_test))
    
    #Boosting Model (Depth=3, T=100)
    b_estimators, b_weights, _, _ = fit_samme_adaboost(X_tr_r, y_tr_r, n_estimators=100, max_depth=3, seed=seed)
    p_boost = get_boosting_proba(b_estimators, b_weights, X_test)
    boosting_probs[r] = p_boost
    boosting_accs.append(np.mean(np.argmax(p_boost, axis=1) == y_test))

    logger.info(f"Single Tree Acc: {single_tree_accs[-1]:.4f}")
    logger.info(f"Bagging Acc:     {bagging_accs[-1]:.4f}")
    logger.info(f"Boosting Acc:    {boosting_accs[-1]:.4f}")

b_st, var_st = compute_bias_and_variance(single_tree_probs, e_targets)
b_bag, var_bag = compute_bias_and_variance(bagging_probs, e_targets)
b_boost, var_boost = compute_bias_and_variance(boosting_probs, e_targets)

logger.info("=" * 85)
logger.info(f"{'Selected Single Tree'} | Bias: {b_st:.4f} | Var: {var_st:.4f} | Mean Acc: {np.mean(single_tree_accs):.4f} | Std Acc: {np.std(single_tree_accs):.4f}")
logger.info(f"{'Final Bagging Model'} | Bias: {b_bag:.4f} | Var: {var_bag:.4f} | Mean Acc: {np.mean(bagging_accs):.4f} | Std Acc: {np.std(bagging_accs):.4f}")
logger.info(f"{'Selected Boosting Model'} | Bias: {b_boost:.4f} | Var: {var_boost:.4f} | Mean Acc: {np.mean(boosting_accs):.4f} | Std Acc: {np.std(boosting_accs):.4f}")
logger.info("=" * 85 + "\n")

# df_a = pd.DataFrame({
#     "Model": ["Selected Single Tree", "Final Bagging Model", "Selected Boosting Model"],
#     "Bias Squared": [b_st, b_bag, b_boost],
#     "Variance": [var_st, var_bag, var_boost],
#     "Mean Test Acc": [np.mean(single_tree_accs), np.mean(bagging_accs), np.mean(boosting_accs)],
#     "Std Test Acc": [np.std(single_tree_accs), np.std(bagging_accs), np.std(boosting_accs)]
# })

X_train: (240, 64, 64), y_train: (240,)
X_val:   (80, 64, 64), y_val:   (80,)
X_test:  (80, 64, 64), y_test:  (80,)
X_test_corrupted:  (80, 64, 64), y_test:  (80,)


KeyboardInterrupt: 